## 📦 1. 环境准备与数据加载

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 通过 rc 参数设置中文字体
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载 exercise 数据集：用户运动实验数据
df = sns.load_dataset("exercise")
print(df.head())
print(f"\n数据规模：{df.shape}")
print(f"\n运动类型：{df['kind'].unique()}")
print(f"\n时间点：{df['time'].unique()}")

> **💡 数据集简介：**
> `exercise` 收录了 **15 名用户（id）在 3 种运动状态（kind）下的脉搏（pulse）数据**，包含 4 个字段：
> - `id`：用户编号（15 人）
> - `kind`：运动类型（rest 休息 / walking 走路 / running 跑步）
> - `time`：时间分钟（1 / 15 / 30 分钟）
> - `pulse`：脉搏跳动次数
>
> 这是理解分面网格（FacetGrid）最经典的入门数据集！

## 📊 2. 基础分面网格图：一行代码拆分维度

In [ ]:
# 📊 基础分面网格图
g = sns.catplot(
    data=df,
    x="time",        # x 轴：时间
    y="pulse",       # y 轴：脉搏
    hue="kind",      # 颜色：运动类型
    col="kind",      # 🔥 核心！按运动类型拆分列
    kind="point",    # 图表类型：点线图
    height=4,        # 每个子图高度
    aspect=0.8,      # 宽高比
    palette="Set1"   # 配色方案
)

# Matplotlib 精细美化
g.figure.suptitle("不同运动类型的脉搏变化对比", fontsize=16, fontweight="bold", y=1.05)
g.set_axis_labels("时间（分钟）", "脉搏")

# 🔥 图例自动外置，不遮挡图表
plt.savefig("facet_grid_basic.png", dpi=300, bbox_inches="tight")
print("✅ 基础分面网格图已保存为 'facet_grid_basic.png'")
plt.show()

**✅ 完成！**

你没有看错，核心代码就这几行：
* `sns.catplot()` 是 Seaborn 的分面网格入口
* `col="kind"` 自动按运动类型拆分成 3 列子图
* `kind="point"` 画出点线趋势图
* `hue="kind"` 用颜色区分运动类型
* `height` 和 `aspect` 控制子图尺寸

> **🤔 什么是分面网格（FacetGrid）？**
> 简单理解：**按某个维度自动拆分图表，排列成网格**。
>
> 就像 Excel 的数据透视表，你选一个字段（如"运动类型"），它自动分成 3 个子表。
> 分面网格是**可视化版本**——自动分成 3 个子图，排列整齐。

## 🚀 3. 进阶玩法：双维度拆分 + 全局标题

In [ ]:
# 🚀 高级玩法:行列双维度拆分
# exercise 数据集没有性别列，我们模拟添加一个

# 模拟添加性别列（奇数 id 为男性，偶数 id 为女性）
df_with_sex = df.copy()
df_with_sex["sex"] = df_with_sex["id"].apply(lambda x: "男" if int(x) % 2 == 0 else "女")
print("添加性别列后数据预览：")
print(df_with_sex.head(10))
print(f"\n性别分布：{df_with_sex['sex'].value_counts().to_dict()}")

In [ ]:
# 🚀 现在用运动类型 × 性别演示双维度分面网格
g = sns.catplot(
    data=df_with_sex,
    x="time",
    y="pulse",
    hue="kind",
    col="kind",      # 列维度：运动类型
    row="sex",       # 行维度：性别
    kind="point",
    height=3.5,
    aspect=1,
    palette="viridis",
    sharey=True      # y 轴共享，方便男女对比
)

# 注入全局大标题
g.figure.suptitle("运动类型 × 性别：脉搏变化多维对比",
               fontsize=18, fontweight="bold", y=1.02)

# 统一设置坐标轴标签
g.set_axis_labels("时间（分钟）", "脉搏（次/分）")

# 🔥 核心：把图例移到图表外面，避免遮挡
g.add_legend(title="运动类型", bbox_to_anchor=(1.02, 0.5), loc="center left")

# 调整右侧边距，为图例留出空间
g.figure.subplots_adjust(right=0.85)

plt.savefig("facet_grid_advanced.png", dpi=300, bbox_inches="tight")
print("✅ 高级分面网格图已保存为 'facet_grid_advanced.png'")
plt.show()

**✅ 搞定！** 现在图表变成了 2×3 的多维网格：
* **列拆分** = `col="kind"` 按运动类型分成 3 列
* **行拆分** = `row="sex"` 按性别分成 2 行
* **颜色区分** = `hue="kind"` 用颜色叠加维度
* **共享 y 轴** = `sharey=True` 方便男女直接对比

> **💡 `catplot` vs `FacetGrid` 的关系：**
* `sns.catplot()` = **高级封装**，一行代码搞定常见分面场景
* `sns.FacetGrid()` = **底层 API**，完全自定义每个子图
* 关系就像 `sns.barplot()` 和 `ax.bar()`，一个快一个灵活
* **90% 场景用 `catplot` 就够了**，剩下 10% 极端定制再用 `FacetGrid`

## 🧪 4. A/B 测试场景:按用户分组对比行为指标

In [ ]:
# 🧪 A/B 测试场景:按用户分组对比行为指标
# 假设你有一份新功能上线前后的用户活跃度数据

# 模拟 A/B 测试数据（实际业务中从数据库读取）
np.random.seed(42)
n_users = 100
ab_df = pd.DataFrame({
    "user_id": range(n_users),
    "group": np.random.choice(["对照组 A", "实验组 B"], n_users),
    "活跃度_第1天": np.random.randint(10, 50, n_users),
    "活跃度_第7天": np.random.randint(5, 60, n_users),
    "活跃度_第30天": np.random.randint(3, 70, n_users)
})

# 转换长格式（Tidy Data）
ab_long = ab_df.melt(
    id_vars=["user_id", "group"],
    value_vars=["活跃度_第1天", "活跃度_第7天", "活跃度_第30天"],
    var_name="时间",
    value_name="活跃度"
)

# 简化时间标签
ab_long["时间"] = ab_long["时间"].str.replace("活跃度_第", "第").str.replace("天", "天")

print("A/B 测试数据预览：")
print(ab_long.head())
print(f"\n数据规模：{ab_long.shape}")

In [ ]:
# 画分面网格
g = sns.catplot(
    data=ab_long,
    x="时间",
    y="活跃度",
    hue="group",        # 颜色区分 A/B 组
    col="group",        # 按 A/B 组拆分列
    kind="point",
    height=4,
    aspect=0.9,
    palette=["#2ecc71", "#e74c3c"],  # 绿 vs 红
    errorbar="sd"       # 显示标准差
)

g.figure.suptitle("A/B 测试：新功能对用户活跃度的影响",
               fontsize=16, fontweight="bold", y=1.05)
g.set_axis_labels("观测时间", "活跃度评分")

# 图例已自动外置，不会遮挡图表
plt.savefig("ab_test_dashboard.png", dpi=300, bbox_inches="tight")
print("✅ A/B 测试看板已保存为 'ab_test_dashboard.png'")
plt.show()

**✅ 效果拉满！** 一张图清晰对比：
* **对照组 A** vs **实验组 B** 的活跃度变化趋势
* 误差线显示标准差，一眼看出波动是否显著
* 红绿配色天然适合 A/B 对比场景

> **🤫 业务洞察：**
> 观察图表，如果实验组 B 的活跃度曲线明显高于对照组 A，且误差线不重叠——
> 恭喜！你的新功能**很可能**真的有效。
>
> 但最终结论还需要统计检验（t 检验、p 值）来背书，图表只是第一步。

## 🔧 5. 底层 API:完全自定义每个子图

In [ ]:
# 🔧 底层 API:完全自定义每个子图
from seaborn import FacetGrid

# 使用原始 exercise 数据
g = FacetGrid(
    data=df,
    col="kind",
    height=4,
    aspect=0.8,
    sharey=True     # y 轴共享，方便对比
)

# 自定义每个子图的画法
g.map(plt.plot, "time", "pulse", marker="o", linewidth=2, markersize=8)
g.map(plt.fill_between, "time", "pulse", alpha=0.1)  # 添加半透明填充

# 设置标题和标签
g.figure.suptitle("自定义分面网格：脉搏趋势对比",
               fontsize=16, fontweight="bold", y=1.05)
g.set_axis_labels("时间（分钟）", "脉搏")
g.set_titles(col_template="{col_name} 运动")  # 自定义子图标题

# 添加图例（外置）
g.add_legend(title="运动类型", bbox_to_anchor=(1.02, 0.5), loc="center left")

# 调整右侧边距
g.figure.subplots_adjust(right=0.85)

plt.savefig("facetgrid_custom.png", dpi=300, bbox_inches="tight")
print("✅ 自定义分面网格图已保存为 'facetgrid_custom.png'")
plt.show()

**✅ 效果：**
* `g.map()` = 对每个子图执行任意 Matplotlib 函数
* `col_template` = 自定义子图标题格式
* `g.add_legend()` = 手动添加全局图例
* **完全控制**每个细节，适合出版级定制

## 💾 6. 完整代码模板:分面网格 A/B 测试看板

In [ ]:
# 💾 完整代码模板:分面网格 A/B 测试看板
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载数据（可替换为你自己的 CSV）
# df = pd.read_csv("your_ab_test.csv")
df = sns.load_dataset("exercise")

# 画分面网格
g = sns.catplot(
    data=df,
    x="time",          # x 轴
    y="pulse",         # y 轴
    hue="kind",        # 颜色维度
    col="kind",        # 列拆分维度
    kind="point",      # 图表类型：point/box/violin/bar...
    height=4,          # 子图高度
    aspect=0.8,        # 子图宽高比
    palette="Set1",    # 配色方案
    errorbar="sd"      # 误差线类型
)

# Matplotlib 全局标题注入
g.figure.suptitle("A/B 测试：多维对比看板", fontsize=16, fontweight="bold", y=1.05)
g.set_axis_labels("时间", "指标值")

# 导出高清图
plt.savefig("facet_grid_dashboard.png", dpi=300, bbox_inches="tight")
print("✅ 看板已保存为 'facet_grid_dashboard.png'")
plt.show()

## 💡 本节核心总结

| 步骤 | 工具 | 核心 API | 作用 |
|------|------|----------|------|
| 基础分面 | Seaborn | `sns.catplot(col=xxx, kind="point")` | 一行代码拆分维度 |
| 双维度拆分 | Seaborn | `col=xxx, row=yyy` | 行列组合生成网格 |
| 全局标题 | Matplotlib | `g.figure.suptitle()` | 注入大标题 |
| 自定义子图 | Seaborn | `g.map(plt.xxx)` | 对每个子图执行任意操作 |
| 长格式转换 | Pandas | `df.melt()` | 宽表转长表 |
| 坐标轴独立 | Seaborn | `sharey=False` | 每个子图自动缩放 |

**给你的练习建议：**
1. 📝 换成 `sns.load_dataset("penguins")`，用 `col="species", row="island"` 生成企鹅体征对比网格。
2. 🎨 把 `palette="Set1"` 改成 `"viridis"`, `"pastel"`, `"colorblind"`，感受配色差异。
3. 🔄 试试 `kind="box"` 或 `kind="violin"` 替换 `"point"`，画出不同类型的分面网格。
4. 📊 用你自己的业务数据（如 A/B 测试、用户分层对比），画出一个真正的商业决策看板。